In [20]:
import sqlite3

conn = sqlite3.connect("credit_card.db")

In [21]:
import pandas as pd


In [22]:
df = pd.read_csv(r"D:\Credit card project\data\clean\clean_transactions.csv")

In [23]:
df_sql = df.copy()

In [24]:
df_sql["age_group"] = df_sql["age_group"].astype(str)

In [25]:
df_sql["age_group"].head(5)

0    Under 26
1       36-50
2       36-50
3       36-50
4         50+
Name: age_group, dtype: str

In [26]:
df_sql["date"] = pd.to_datetime(df_sql["date"]).dt.strftime("%Y-%m-%d")
df_sql["birthdate"] = pd.to_datetime(df_sql["birthdate"]).dt.strftime("%Y-%m-%d")
df_sql["age_group"] = df_sql["age_group"].astype(str)


In [27]:
df_sql.to_sql("transactions", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM transactions", conn))

   total_rows
0       50000


In [28]:
print(df.columns.tolist())

['customer_id', 'name', 'surname', 'gender', 'birthdate', 'transaction_amount', 'date', 'merchant_name', 'category', 'month', 'weekday', 'is_weekend', 'age', 'age_group', 'high_value']


monthly spend

In [29]:
q1 = """
SELECT month,
       COUNT(*) AS transactions,
       ROUND(SUM(transaction_amount), 2) AS total_spend
FROM transactions
GROUP BY month
ORDER BY month;
"""
print(pd.read_sql(q1, conn))

     month  transactions  total_spend
0  2023-01          5444   2345228.21
1  2023-02          4854   2148030.95
2  2023-03          5456   2396963.31
3  2023-04          5091   2289731.30
4  2023-05          5493   2460816.56
5  2023-06          5258   2254592.41
6  2023-07          5458   2475041.53
7  2023-08          5399   2391450.20
8  2023-09          5153   2274245.66
9  2023-10          2394   1069861.84


cateogry wise share percent

In [30]:
q2 = """
SELECT category,
       ROUND(SUM(transaction_amount), 2) AS total_spend,
       ROUND(SUM(transaction_amount) * 100.0 /
             (SELECT SUM(transaction_amount) FROM transactions), 2) AS share_pct
FROM transactions
GROUP BY category
ORDER BY total_spend DESC;
"""
print(pd.read_sql(q2, conn))

      category  total_spend  share_pct
0       Travel  12900231.94      58.36
1  Electronics   4394092.10      19.88
2       Market   2151134.42       9.73
3     Clothing   1319342.47       5.97
4     Cosmetic    876672.44       3.97
5   Restaurant    464488.60       2.10


spending according to age group

In [31]:
q3 = """
SELECT age_group,
       COUNT(*) AS transactions,
       ROUND(AVG(transaction_amount), 2) AS avg_ticket
FROM transactions
GROUP BY age_group
ORDER BY avg_ticket DESC;
"""
print(pd.read_sql(q3, conn))

  age_group  transactions  avg_ticket
0       50+         20843      445.43
1     26-35          8531      441.86
2     36-50         12706      440.71
3  Under 26          7920      435.94


spend according to genders

In [32]:
q4 = """
SELECT gender,
       COUNT(*) AS transactions,
       ROUND(SUM(transaction_amount), 2) AS total_spend,
       ROUND(AVG(transaction_amount), 2) AS avg_ticket
FROM transactions
GROUP BY gender
ORDER BY total_spend DESC;
"""
print(pd.read_sql(q4, conn))

    gender  transactions  total_spend  avg_ticket
0        F         22713  10119120.24      445.52
1        M         22240   9794882.81      440.42
2  Unknown          5047   2191958.92      434.31


Weekend vs Weekday

In [33]:
q5 = """
SELECT is_weekend,
       COUNT(*) AS transactions,
       ROUND(SUM(transaction_amount), 2) AS total_spend,
       ROUND(AVG(transaction_amount), 2) AS avg_ticket
FROM transactions
GROUP BY is_weekend;
"""
print(pd.read_sql(q5, conn))

   is_weekend  transactions  total_spend  avg_ticket
0           0         35731  15817409.04      442.68
1           1         14269   6288552.93      440.71


in which cateogry high value transaction belong

In [34]:
q6 = """
SELECT category,
       COUNT(*) AS high_value_txns,
       ROUND(SUM(transaction_amount), 2) AS high_value_spend
FROM transactions
WHERE high_value = 1
GROUP BY category
ORDER BY high_value_txns DESC;
"""
print(pd.read_sql(q6, conn))

      category  high_value_txns  high_value_spend
0   Restaurant              421          41177.65
1       Market              420         204692.39
2       Travel              419        1222951.91
3  Electronics              417         407681.38
4     Clothing              414         121400.60
5     Cosmetic              413          80698.27
